# Scholarship Scheme Ingest → Delta Table

Reads scholarship scheme CSV files from UC Volume `/Volumes/main/scholarships/raw/`
and writes them to the Delta table `main.scholarships.scheme_corpus` with a
rich `text` column ready for embedding.

**Prerequisites:**
- Unity Catalog schema `main.scholarships` exists.
- UC Volume `main.scholarships.raw` exists with CSV files uploaded.

**Expected CSV columns:**
`scheme_id`, `scheme_name`, `administering_body`, `level` (Central/State),
`state`, `eligible_categories`, `min_income_limit`, `max_income_limit`,
`eligible_gender`, `eligible_education_levels`, `eligible_disability`,
`eligible_minority`, `award_amount`, `application_deadline`,
`description_text`, `source_url`

Run this notebook **before** `setup_vector_search.ipynb`.

In [ ]:
CATALOG = "main"
SCHEMA = "scholarships"
TABLE = "scheme_corpus"
RAW_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

print(f"Source: {RAW_VOLUME}")
print(f"Target: {CATALOG}.{SCHEMA}.{TABLE}")

In [ ]:
# Create schema and volumes if not already present.
# Run once — safe to re-run (IF NOT EXISTS).
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.rag")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")
print(f"Schema and volumes ready: {CATALOG}.{SCHEMA}")

In [ ]:
import os
import glob

csv_files = glob.glob(os.path.join(RAW_VOLUME, "*.csv"))
print(f"Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(" ", f)

In [ ]:
import pandas as pd

EXPECTED_COLS = [
    "scheme_id", "scheme_name", "administering_body", "level", "state",
    "eligible_categories", "min_income_limit", "max_income_limit",
    "eligible_gender", "eligible_education_levels", "eligible_disability",
    "eligible_minority", "award_amount", "application_deadline",
    "description_text", "source_url",
]

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    missing = [c for c in EXPECTED_COLS if c not in df.columns]
    if missing:
        print(f"WARNING: {f} is missing columns: {missing}")
    dfs.append(df)

if not dfs:
    raise RuntimeError(f"No CSV files found at {RAW_VOLUME}. Upload your scholarship CSVs first.")

raw_df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(raw_df)} rows from {len(dfs)} file(s)")
raw_df.head(3)

In [ ]:
def _safe(val, default=""):
    """Return str(val) or default for None/NaN."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return default
    return str(val).strip()


def build_text(row) -> str:
    """Concatenate scheme fields into a rich text string for embedding."""
    return (
        f"Scheme: {_safe(row.get('scheme_name'))}. "
        f"Administered by: {_safe(row.get('administering_body'))} "
        f"({_safe(row.get('level'))}). "
        f"Eligible categories: {_safe(row.get('eligible_categories'))}. "
        f"Income limit: below \u20b9{_safe(row.get('max_income_limit'))}. "
        f"Education: {_safe(row.get('eligible_education_levels'))}. "
        f"Gender: {_safe(row.get('eligible_gender'))}. "
        f"Disability: {_safe(row.get('eligible_disability'))}. "
        f"Minority: {_safe(row.get('eligible_minority'))}. "
        f"Award: {_safe(row.get('award_amount'))}. "
        f"Deadline: {_safe(row.get('application_deadline'))}. "
        f"Details: {_safe(row.get('description_text'))}"
    )


raw_df["text"] = raw_df.apply(build_text, axis=1)
print("Sample text column:")
print(raw_df["text"].iloc[0])

In [ ]:
# Write to Delta table
sdf = spark.createDataFrame(raw_df)
(
    sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")
)
print(f"\n✅ Written {len(raw_df)} rows to {CATALOG}.{SCHEMA}.{TABLE}")
spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}").select(
    "scheme_id", "scheme_name", "level", "state", "award_amount"
).show(5, truncate=60)

In [ ]:
# Smoke test: read back and verify
count = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}").count()
print(f"Row count in Delta table: {count}")
assert count == len(raw_df), f"Expected {len(raw_df)}, got {count}"
print("✅ Ingest complete — ready for setup_vector_search.ipynb")